# Instacart benchmark — thesis-aligned notebook

Includes training, evaluation, Diebold–Mariano tests, Holm correction, 1,000-resample weekly-cluster bootstrap intervals, and three-seed prediction-based variability summaries.

**Reproduction note:** The multi-seed summary loads prediction files for seeds 42, 123 and 456; it does not automatically train the additional seeds. Run the final training and evaluation cells separately for each seed, retaining fixed hyperparameters and identical data splits, before executing the summary. See `docs/REPRODUCIBILITY.md`.

Notebook outputs have been cleared for public release. The industrial dataset is not distributed.


# Level 0 - Introduction

## 1) Import Libraries

In [ ]:
#import libraries
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from scipy import sparse
import networkx as nx
from networkx.algorithms.community import greedy_modularity_communities
import torch
import torch.nn as nn
import torch.nn.functional as F
from scipy.stats import t
from pathlib import Path
import json
import pickle


# Install torch_geometric if not already installed
try:
    import torch_geometric
except ImportError:
    print("Installing torch-geometric...")
    raise ImportError("Install the pinned project dependencies from requirements.txt before running this notebook")
    import torch_geometric

from torch_geometric.nn import GCNConv, SAGEConv
from torch_geometric.data import Data
import torch.optim as optim
import random
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
import warnings
import platform


# folder where figures will be saved
FIG_DIR = "figures"
os.makedirs(FIG_DIR, exist_ok=True)
def save_fig(name):
 path = os.path.join(FIG_DIR, name)
 plt.savefig(path, dpi=300, bbox_inches="tight")
 print("Saved:", path)


In [ ]:
# ---Global seed initializer ---
import random
import numpy as np
import torch


EXPERIMENT_SEED = 42
def seed_everything(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    # Force deterministic operations within PyTorch operations
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

# Execute the seed setting
seed_everything(EXPERIMENT_SEED)
print("Global seeding locked successfully.")

## 1.2 Reproducibility and Software Environment

In [ ]:
# =============================================================================
# REPRODUCIBILITY AND SOFTWARE ENVIRONMENT DOCUMENTATION
# =============================================================================

import sys
import platform
import torch
import pandas as pd
import numpy as np
import networkx as nx
import statsmodels
import sklearn

try:
    import torch_geometric
    pyg_version = torch_geometric.__version__
except Exception:
    pyg_version = "Not installed / not detected"

environment_info = {
    "Python version": sys.version.split()[0],
    "Platform": platform.platform(),
    "PyTorch version": torch.__version__,
    "PyTorch Geometric version": pyg_version,
    "CUDA available": torch.cuda.is_available(),
    "CUDA version": torch.version.cuda,
    "GPU device": torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU only",
    "Pandas version": pd.__version__,
    "NumPy version": np.__version__,
    "NetworkX version": nx.__version__,
    "Statsmodels version": statsmodels.__version__,
    "Scikit-learn version": sklearn.__version__,
    "Random seed": EXPERIMENT_SEED,
}

environment_df = pd.DataFrame(
    list(environment_info.items()),
    columns=["Component", "Version / Setting"]
)

display(environment_df)

# 3.3 - Data Sources and Data Description

In [ ]:

# CHANGE THIS to the folder where you extracted the Kaggle dataset
# Example: DATA_DIR = r"C:\Users\you\Downloads\instacart_2017_05_01"
from google.colab import drive
drive.mount('/content/drive/')

from pathlib import Path
import json
import pickle

DATASET_NAME = "benchmark_instacart"

PROJECT_DIR = Path(
    "/content/drive/MyDrive/FMCG_Forecasting"
) / DATASET_NAME

CHECKPOINT_DIR = PROJECT_DIR / "checkpoints"
PREDICTION_DIR = PROJECT_DIR / "predictions"
HISTORY_DIR = PROJECT_DIR / "histories"
METRICS_DIR = PROJECT_DIR / "metrics"
CONFIG_DIR = PROJECT_DIR / "config"
ROBUSTNESS_DIR = PROJECT_DIR / "robustness"

for directory in [
    CHECKPOINT_DIR,
    PREDICTION_DIR,
    HISTORY_DIR,
    METRICS_DIR,
    CONFIG_DIR,
    ROBUSTNESS_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

DATA_DIR = "/content/drive/MyDrive/archive"
# Defensive check: confirm folder exists before proceeding
assert os.path.isdir(DATA_DIR), f"DATA_DIR not found: {DATA_DIR}"




# Load orders: contains one row per order with user ordering sequence and time gap proxy
orders_path = os.path.join(DATA_DIR, "orders.csv")
orders = pd.read_csv(orders_path)
# Load order_products__prior: contains the products that appear in each historical order
# This is the key "order–product association table" used for:
# - demand aggregation (counts by product over time)
# - co-purchase edge creation (products appearing together in the same order)
prior_path = os.path.join(DATA_DIR, "order_products__prior.csv")
order_products_prior = pd.read_csv(prior_path)
# Load products: product metadata, helps interpretation and enables optional categorical features later
products_path = os.path.join(DATA_DIR, "products.csv")
products = pd.read_csv(products_path)
# OPTIONAL: load aisles and departments for later interpretability (not used yet in Level 1 computations)
aisles_path = os.path.join(DATA_DIR, "aisles.csv")
departments_path = os.path.join(DATA_DIR, "departments.csv")
aisles = pd.read_csv(aisles_path) if os.path.exists(aisles_path) else None
departments = pd.read_csv(departments_path) if os.path.exists(departments_path) else None
# OPTIONAL: Kaggle "train" table (not used unless you explicitly choose to merge for extra transactions later)
train_path = os.path.join(DATA_DIR, "order_products__train.csv")
order_products_train = pd.read_csv(train_path) if os.path.exists(train_path) else None

# Basic integrity checks:
# - Ensure key identifiers exist
# - Ensure we have expected row counts and basic column structure
print("orders shape:", orders.shape)
print("order_products_prior shape:", order_products_prior.shape)
print("products shape:", products.shape)
required_orders_cols = {"order_id", "user_id", "order_number", "days_since_prior_order"}
required_prior_cols = {"order_id", "product_id"}
required_products_cols = {"product_id", "product_name", "aisle_id", "department_id"}
missing_orders = required_orders_cols - set(orders.columns)
missing_prior = required_prior_cols - set(order_products_prior.columns)
missing_products = required_products_cols - set(products.columns)
assert not missing_orders, f"orders.csv missing columns: {missing_orders}"
assert not missing_prior, f"order_products__prior.csv missing columns: {missing_prior}"
assert not missing_products, f"products.csv missing columns: {missing_products}"
# Display a few rows for verification (helpful during defense demonstrations)
display(orders.head())
display(order_products_prior.head())
display(products.head())

# 3.4. Data Processing and Cleaning

## 1)Synthetic Timestamp Built

In [ ]:
# Instacart provides order sequence and days_since_prior_order, not real timestamps.
# To support chronological splitting and time-series aggregation, a pseudo-timestamp reconstruct.
orders = orders.copy()
# 1) Ensure correct ordering per user
orders = orders.sort_values(["user_id", "order_number"])
# 2) Replace missing days_since_prior_order for a user's first order with 0
# (First order has no prior gap; this prevents NaNs from breaking the cumulative sum.)
orders["days_since_prior_order"] = orders["days_since_prior_order"].fillna(0)
# 3) Cumulative days since user's first observed order
orders["cum_days"] = orders.groupby("user_id")["days_since_prior_order"].cumsum()
# 4) Map cumulative days to a synthetic calendar date
# NOTE: The absolute date is arbitrary; what matters is correct ordering and spacing.
START_DATE = pd.Timestamp("2016-01-01")
orders["order_ts"] = START_DATE + pd.to_timedelta(orders["cum_days"], unit="D")
# Sanity check: the timestamp should be non-decreasing within each user
assert (orders.groupby("user_id")["order_ts"].diff().fillna(pd.Timedelta(0)) >= pd.Timedelta(0)).all()
display(orders.head())


In [ ]:
# =============================================================================
# DATA QUALITY ASSESSMENT — BEFORE CLEANING
# =============================================================================

def data_quality_report(name, df):
    print("="*80)
    print(f"DATA QUALITY REPORT: {name}")
    print("="*80)
    print("Shape:", df.shape)
    print("\nMissing values:")
    print(df.isnull().sum())
    print("\nDuplicate full rows:", df.duplicated().sum())
    print("\nData types:")
    print(df.dtypes)
    print()

data_quality_report("orders", orders)
data_quality_report("products", products)
data_quality_report("order_products_prior", order_products_prior)

# Specific duplicate order-product checks
duplicate_order_products = order_products_prior.duplicated(
    subset=["order_id", "product_id"]
).sum()

print("Duplicate order-product pairs:", duplicate_order_products)

# Referential integrity checks
missing_order_refs = (~order_products_prior["order_id"].isin(orders["order_id"])).sum()
missing_product_refs = (~order_products_prior["product_id"].isin(products["product_id"])).sum()

print("Order-product rows with missing order reference:", missing_order_refs)
print("Order-product rows with missing product reference:", missing_product_refs)

## 2) Data cleaning

In [ ]:
# =============================================================================
# DATA CLEANING
# =============================================================================

before_shape = order_products_prior.shape

order_products_prior = order_products_prior.copy()

# Remove rows with missing critical identifiers
missing_before = order_products_prior[["order_id", "product_id"]].isnull().sum()

order_products_prior = order_products_prior.dropna(
    subset=["order_id", "product_id"]
)

# Convert IDs to integers
order_products_prior["order_id"] = order_products_prior["order_id"].astype(int)
order_products_prior["product_id"] = order_products_prior["product_id"].astype(int)
orders["order_id"] = orders["order_id"].astype(int)
products["product_id"] = products["product_id"].astype(int)

# Remove duplicate product entries within the same order
duplicates_before = order_products_prior.duplicated(
    subset=["order_id", "product_id"]
).sum()

order_products_prior = order_products_prior.drop_duplicates(
    subset=["order_id", "product_id"]
)

after_shape = order_products_prior.shape

print("="*80)
print("DATA CLEANING SUMMARY")
print("="*80)
print("Shape before cleaning:", before_shape)
print("Shape after cleaning :", after_shape)
print("\nMissing IDs before cleaning:")
print(missing_before)
print("\nDuplicate order-product pairs removed:", duplicates_before)
print("Cleaned order_products_prior shape:", order_products_prior.shape)


## 3) Merge tables forms to df

In [ ]:
# Merge order timestamps into order-product rows.
# Output: one row = one product in one order, with an associated timestamp.
# This unified table is the foundation for both:
# - demand aggregation (time series)
# - co-purchase edge extraction (basket analysis)
df = order_products_prior.merge(
 orders[["order_id", "user_id", "order_number", "order_ts"]],
 on="order_id",
 how="inner" # keep only order-product rows whose order exists in orders table
)
# In Instacart, quantity is not provided; each row means the product was purchased once in that order.
df["quantity"] = 1
# Sanity checks
assert df["order_ts"].notna().all()
assert df["product_id"].notna().all()
print("Unified transactional table df shape:", df.shape)
display(df.head())


## 4) TOP-N filtering

In [ ]:
# IMPORTANT COMPUTE NOTE:
# Instacart has ~50k products; full graph construction + GNN training can exceed memory/time.
# So the most frequently purchased products is focus on,
# because they have enough signal for forecasting and stable co-purchase relationships.
# The Top-N products by total purchase count are chosen.
TOP_N_PRODUCTS = 2000 # adjust based on your machine (1000–5000 typical)
top_products = (
 df["product_id"]
 .value_counts()
 .head(TOP_N_PRODUCTS)
 .index
)
df = df[df["product_id"].isin(top_products)].copy()
print("After Top-N product filtering, df shape:", df.shape)
print("Unique products kept:", df["product_id"].nunique())
print("\nComment:")
print("This filter improves feasibility and model stability; very rare")
print("products are too sparse for reliable time-series modeling and add noise to network structure")

## 5) Weekly Aggregation

In [ ]:
# Methodology alignment:
# - Aggregate purchases into fixed temporal intervals (weekly) to reduce sparsity/noise
# while preserving meaningful demand dynamics. :contentReference[oaicite:1]{index=1}
# Create a weekly bucket for each transaction row
df["week"] = df["order_ts"].dt.to_period("W").dt.start_time
# Aggregate demand: weekly purchase counts per product
weekly = (
 df.groupby(["week", "product_id"])["quantity"]
 .sum()
 .reset_index()
 .rename(columns={"quantity": "demand"})
    )
display(weekly.head())
print("Weeks:", weekly["week"].nunique(), "| Products:", weekly["product_id"].nunique())

## 6) Pivoting the Demand Matrix (TXN)

In [ ]:
# Convert to a matrix format:
# rows = time (weeks), columns = products, values = demand
# This format is convenient for creating lagged features and forecast targets.
demand_mat = (
 weekly.pivot(index="week", columns="product_id", values="demand")
 .fillna(0.0)
 .sort_index()
)
print("Demand matrix shape (T weeks x N products):", demand_mat.shape)
display(demand_mat.head())

## 7) Summary of Data Processing and Cleaning

In [ ]:
print("df:", df.shape)
print("weekly:", weekly.shape)
print("demand_mat:", demand_mat.shape)
print("Time range:", demand_mat.index.min(), "to", demand_mat.index.max())
print("Products:", demand_mat.shape[1])

# 3.5 Network Construction and Social Network Analysis
**Goal:** Create a weighted co-purchase graph where:
- Node = product
- Edge = co-purchase relationship
- Edge weight = co-purchase frequency

## 3.5.1. Sparse Matrix incidence + co-occurence C

In [ ]:
# 3.5.1 — FAST CO-PURCHASE COUNTING USING SPARSE MATRICES
# ----------------------------------------------------------
# Goal (methodology alignment):
# - Build a weighted product network where:
# Node = product (SKU)
# Edge = two products co-occur in the same order (co-purchase)
# Edge weight = frequency of co-purchase across all orders
#
# Why sparse matrices:
# - Instacart has millions of orders. Looping through orders and generating all product pairs is too slow.
# - Instead we build a sparse binary matrix X with:
# rows = orders, columns = products
# X[o, p] = 1 if product p appears in order o
# - Then co-purchase counts are obtained by matrix multiplication:
# C = X.T @ X
# where C[i, j] is the number of orders that contained BOTH products i and j.
# - This is mathematically equivalent to counting co-purchases via loops, but far faster and scalable.

# Defensive checks: ensure required columns exist in df (from Level 2)
assert {"order_id", "product_id"}.issubset(df.columns), "df must include order_id and product_id"
# Keep only unique order-product pairs (avoid duplicates inflating co-occurrence counts)
pairs = df[["order_id", "product_id"]].drop_duplicates().copy()
pairs["order_id"] = pairs["order_id"].astype(int)
pairs["product_id"] = pairs["product_id"].astype(int)
# IMPORTANT: Ensure we only use product_ids that exist in demand_mat columns (Top-N already applied)
product_ids = demand_mat.columns.astype(int).to_numpy()
pid_to_col = pd.Series(np.arange(len(product_ids)), index=product_ids)
pairs = pairs[pairs["product_id"].isin(pid_to_col.index)]
# Map order_id -> row index
order_ids = pairs["order_id"].unique()
order_id_to_row = pd.Series(np.arange(len(order_ids)), index=order_ids)
# Convert pairs into row/col indices for sparse matrix creation
row_idx = order_id_to_row.loc[pairs["order_id"]].to_numpy()
col_idx = pid_to_col.loc[pairs["product_id"]].to_numpy()
# Build sparse incidence matrix X (orders x products)
# Each nonzero indicates the product is present in that order
data = np.ones(len(pairs), dtype=np.uint8)
X = sparse.csr_matrix((data, (row_idx, col_idx)), shape=(len(order_ids), len(product_ids)))
print("Incidence matrix X created.")
print("Orders:", X.shape[0], "| Products:", X.shape[1], "| Non-zeros (purchases):", X.nnz)
# Co-occurrence matrix C = X^T X gives co-purchase counts
C = (X.T @ X).astype(np.int32)
# Remove diagonal: product co-occurring with itself is not an edge
C.setdiag(0)
C.eliminate_zeros()
print("Co-occurrence matrix C created.")
print("Non-zero co-occurrences (potential directed edges in sparse form):", C.nnz)

## 3.5.2 Threshold Graph Build

In [ ]:
# CELL 3.5.2 — BUILD THE WEIGHTED PRODUCT GRAPH WITH THRESHOLDING
# -------------------------------------------------------------
# Goal:
# - Convert co-occurrence counts into a network/graph representation:
# Nodes = products
# Edges = co-purchase relationships
# Weights = co-purchase frequency
#
# Thresholding (methodology alignment):
# - Very weak edges (rare co-purchases) can represent noise/randomness rather than stable relationships.
# - Applying a threshold improves interpretability and reduces graph density.

def build_graph_from_C(C, product_ids, min_edge_weight):
 """
 Build an undirected weighted NetworkX graph from sparse co-occurrence matrix C.
 Only keeps edges with weight >= min_edge_weight.
 """
 # Keep only entries meeting the threshold
 C_thr = C.multiply(C >= min_edge_weight)
 C_thr.eliminate_zeros()
 # Convert to coordinate format to iterate over non-zeros efficiently
 C_coo = C_thr.tocoo()
 # Create graph and add all nodes
 G = nx.Graph()
 G.add_nodes_from(product_ids.tolist())
 # Add undirected edges (only keep i < j to avoid duplicates)
 for i, j, w in zip(C_coo.row, C_coo.col, C_coo.data):
      if i < j:
          a = int(product_ids[i])
          b = int(product_ids[j])
          G.add_edge(a, b, weight=int(w))

 return G
# START with a higher threshold than 10 for Instacart, because scale creates many weak co-purchases
MIN_EDGE_WEIGHT = 10 # adjust; we'll evaluate density in Cell 3.3
G = build_graph_from_C(C, product_ids, MIN_EDGE_WEIGHT)
print("Graph built with threshold:", MIN_EDGE_WEIGHT)
print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())


## 3.5.3. Graph Quality Diagnostics


In [ ]:
# CELL 3.3 — GRAPH QUALITY CHECKS (SPARSE VS DENSE)
# -------------------------------------------------
# Why this matters:
# - If the graph is too dense, almost every product is connected to almost every other product.
# Then message passing in a GNN becomes noisy and less meaningful.
# - If the graph is too sparse, it may fragment into many disconnected components and
# relational learning is limited.
#
# For 2000 products:
# - Ideal density is often around 0.01–0.05 for meaningful structure
# - Average degree often in 20–100 range (rule of thumb; depends on threshold)

num_nodes = G.number_of_nodes()
num_edges = G.number_of_edges()
max_edges = num_nodes * (num_nodes - 1) / 2
density = num_edges / max_edges
avg_degree = (2 * num_edges) / num_nodes
print("Nodes:", num_nodes)
print("Edges:", num_edges)
print("Density:", density)
print("Average degree:", avg_degree)

# Connected components
components = list(nx.connected_components(G))
components_sorted = sorted(components, key=len, reverse=True)
lcc = components_sorted[0] # largest connected component nodes
print("Connected components:", len(components))
print("Largest connected component size:", len(lcc), f"({len(lcc)/num_nodes:.1%} of nodes)")
# Edge weight summary (helps justify threshold)
weights = [d["weight"] for _, _, d in G.edges(data=True)]
if weights:
    print("Edge weight distribution (after threshold):")
    print(pd.Series(weights).describe(percentiles=[0.5, 0.75, 0.9, 0.95, 0.99]))
else:
 print("No edges. Lower MIN_EDGE_WEIGHT.")

## 3.5.4.  SNA metrics

In [ ]:
# SNA metrics help characterize structural importance of products in the demand network (methodology).
# We'll compute:
# - degree: number of neighbors (how many products it's connected to)
# - strength: sum of edge weights (how frequently it co-purchases with others)
# - betweenness centrality (approx): bridging importance
# - eigenvector centrality (optional): influence in network
# Degree (unweighted) and Strength (weighted degree)
degree_dict = dict(G.degree())
strength_dict = dict(G.degree(weight="weight"))
# Betweenness can be expensive; use approximation if needed
# Here, with 2000 nodes it's usually okay but can take some time depending on edges.
# If it is slow, we can use nx.betweenness_centrality_subset or approximate sampling.
betweenness = nx.betweenness_centrality(G, k=min(200, num_nodes), seed=42) # approximation using k sample nodes
# Eigenvector centrality can fail if graph disconnected; compute on LCC for stability
G_lcc = G.subgraph(lcc).copy()
eigen_lcc = nx.eigenvector_centrality(G_lcc, max_iter=1000)
# Put metrics into a dataframe
sna = pd.DataFrame({
 "product_id": list(G.nodes()),
 "degree": [degree_dict.get(n, 0) for n in G.nodes()],
 "strength": [strength_dict.get(n, 0.0) for n in G.nodes()],
 "betweenness_approx": [betweenness.get(n, 0.0) for n in G.nodes()],
 "eigenvector_lcc": [eigen_lcc.get(n, np.nan) for n in G.nodes()], # NaN for nodes not in LCC
})
# Join product names (optional but makes defense much stronger)
prod_names = products[["product_id", "product_name", "aisle_id", "department_id"]].copy()
sna = sna.merge(prod_names, on="product_id", how="left")
sna.sort_values("strength", ascending=False).head(10)

## 3.5.5. Community Detection

In [ ]:
# Community structure is used to identify clusters of closely related products (methodology). :contentReference[oaicite:5]{index=5}
# We'll use NetworkX greedy modularity communities (no extra packages needed).

# Community detection usually works best on the LCC
communities = list(greedy_modularity_communities(G_lcc, weight="weight"))

print("Communities found in LCC:", len(communities))
print("Largest 5 community sizes:", sorted([len(c) for c in communities], reverse=True)[:5])

# Assign community IDs
community_id = {}
for i, comm in enumerate(communities):
    for node in comm:
        community_id[node] = i
# Add community labels to SNA table (NaN for nodes not in LCC)
sna["community"] = sna["product_id"].map(community_id)

# Show example: top products in the largest community by strength
largest_comm = max(communities, key=len)
largest_comm_id = community_id[next(iter(largest_comm))]

(
 sna[sna["community"] == largest_comm_id]
 .sort_values("strength", ascending=False)
 .head(15)[["product_id", "product_name", "degree", "strength", "community"]]
)


# 3.6 Graph Representation and Problem Formulaiton
**Goal:** Convert the network + demand time series into PyTorch Geometric snapshots:
- Node features = lagged demand (LAGS)
- Target = next-week demand (HORIZON = 1)
- edge_index / edge_weight from co-purchase network
- Chronological split into train/val/test

## 3.6.1. Final Graph Ready for Modelling

In [ ]:
print("Final graph ready for modeling")
print("Nodes:", G.number_of_nodes())
print("Edges:", G.number_of_edges())
print("Density:", nx.density(G))
print("LCC (Largest Connected Component) nodes:", G_lcc.number_of_nodes())
print("Communities:", len(communities))

## 3.6.2. LCC selection

In [ ]:
# Since thr=10 yields a fully connected graph, the LCC contains all nodes.
G_lcc = G.copy()
lcc_nodes = sorted(list(G_lcc.nodes()))
print("Nodes in modeling graph:", len(lcc_nodes)) # expect 2000
demand_lcc = demand_mat[lcc_nodes].copy()
print("demand_lcc shape (T x N):", demand_lcc.shape) # expect (53, 2000)

## 3.6.3 Node Mapping

In [ ]:
product_ids = np.array(lcc_nodes, dtype=int)
pid_to_idx = {pid: i for i, pid in enumerate(product_ids)}
idx_to_pid = {i: pid for pid, i in pid_to_idx.items()}
print("Example mapping:", product_ids[0], "->", pid_to_idx[product_ids[0]])

## 3.6.4. Edget_index and Edge_weight

In [ ]:
src, dst, wts = [], [], []
for a, b, d in G_lcc.edges(data=True):
 i = pid_to_idx[int(a)]
 j = pid_to_idx[int(b)]
 w = float(d.get("weight", 1.0))
 # add both directions
 src += [i, j]
 dst += [j, i]
 wts += [w, w]
edge_index = torch.tensor([src, dst], dtype=torch.long)
edge_weight = torch.tensor(wts, dtype=torch.float32)
print("edge_index shape:", edge_index.shape) # should be (2, 2*63269) = (2, 126538)
print("edge_weight shape:", edge_weight.shape) # should be (126538,)


## 3.6.5. Lags and Horizon

In [ ]:
LAGS = 8
HORIZON = 1
T, N = demand_lcc.shape
assert T > (LAGS + HORIZON), "Not enough time steps for chosen LAGS/HORIZON."
print("T:", T, "| N:", N, "| LAGS:", LAGS, "| HORIZON:", HORIZON)

## 3.6.6. Snapshots

In [ ]:
X = demand_lcc.values.astype(np.float32) # (T, N)
times = demand_lcc.index
snapshots = []
for t in range(LAGS, T - HORIZON):
 x_t = X[t - LAGS:t, :].T # (N, LAGS)
 y_t = X[t + HORIZON, :] # (N,)
 data = Data(
 x=torch.from_numpy(x_t),
 y=torch.from_numpy(y_t),
 edge_index=edge_index,
 edge_weight=edge_weight
 )
 data.time = times[t]
 snapshots.append(data)
print("Snapshots:", len(snapshots)) # should be 53 - 8 - 1 = 44
print("First snapshot time:", snapshots[0].time)

## 3.6.7 Chronological Splits

In [ ]:
TRAIN_RATIO = 0.70
VAL_RATIO = 0.15
n = len(snapshots)
train_end = int(n * TRAIN_RATIO)

val_end = int(n * (TRAIN_RATIO + VAL_RATIO))
train_data = snapshots[:train_end]
val_data = snapshots[train_end:val_end]
test_data = snapshots[val_end:]
print("Train/Val/Test:", len(train_data), len(val_data), len(test_data))
print("Train range:", train_data[0].time, "to", train_data[-1].time)
print("Test range:", test_data[0].time, "to", test_data[-1].time)

### 3.6.8 Snapshot Diagnostics

In [ ]:
print("Feature shape:", snapshots[0].x.shape)
print("Target shape:", snapshots[0].y.shape)

print("Feature mean:", snapshots[0].x.mean().item())
print("Feature std:", snapshots[0].x.std().item())

print("Target mean:", snapshots[0].y.mean().item())
print("Target std:", snapshots[0].y.std().item())

print("Train:", len(train_data))
print("Val:", len(val_data))
print("Test:", len(test_data))

## 3.6.9 Summary

In [ ]:
ex = train_data[0]
print("x shape:", ex.x.shape) # expect (1978, 8)
print("y shape:", ex.y.shape) # expect (1978,)
print("edge_index shape:", ex.edge_index.shape)
print("edge_weight shape:", ex.edge_weight.shape)
print("time:", ex.time)

### 3.7 — Baseline Models

We implement baselines to benchmark the proposed GNN models:

- **ARIMA** (statistical time-series baseline): forecasts aggregated total weekly demand
- **MLP** (deep learning baseline): forecasts product-level demand using lag features only (no graph)

#### 3.9.2.1.1 - Baseline Model (ARIMA)

*Baseline Model: ARIMA*

To benchmark the proposed graph-based forecasting model, a classical statistical time-series method is implemented.

ARIMA models temporal dependencies but does not capture inter-product relationships.
Therefore it provides an appropriate baseline for evaluating the added value
of graph neural networks.

##### 1) Aggregate accross all products

In [ ]:
# Aggregate demand across all products
total_demand = demand_mat.sum(axis=1)

print("Total demand series length:", len(total_demand))
total_demand.head()

##### 2) Chronological Splits

In [ ]:
T = len(total_demand)

train_end = int(T * 0.70)
val_end = int(T * 0.85)

train_ts = total_demand[:train_end]
val_ts = total_demand[train_end:val_end]
test_ts = total_demand[val_end:]

print("Train:", len(train_ts))
print("Validation:", len(val_ts))
print("Test:", len(test_ts))

##### 3) Fit the ARIMA Model

In [ ]:
# Simple ARIMA configuration
# (2,1,2) works well for many demand series

train_ts_fixed = train_ts.copy()
if hasattr(train_ts_fixed, 'index') and pd.api.types.is_datetime64_any_dtype(train_ts_fixed.index):
    train_ts_fixed.index = pd.to_datetime(train_ts_fixed.index)
    train_ts_fixed = train_ts_fixed.asfreq('W-MON')  # Explicitly set weekly Monday frequency

# ARIMA configuration with better parameter handling
arima = ARIMA(
    train_ts_fixed,
    order=(2, 1, 2),
    # Add these parameters to handle the non-stationary/non-invertible warnings
    enforce_stationarity=False,  # Allow non-stationary AR parameters during fitting
    enforce_invertibility=False,  # Allow non-invertible MA parameters during fitting
    concentrate_scale=True  # Improves numerical stability
)

# Fit the model with corrected optimization settings
arima_model = arima.fit()  # Removed method parameter - let it use the default optimization method
# The default method is usually more stable and handles edge cases better

print(arima_model.summary())

##### 4)Forecast Test Period

In [ ]:
forecast = arima_model.forecast(steps=len(test_ts))

forecast.head()

##### 5) Evaluate the ARIMA Model

In [ ]:
def mae(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))

def rmse(y_true, y_pred):
    return np.sqrt(np.mean((y_true - y_pred)**2))

arima_mae = mae(test_ts.values, forecast.values)
arima_rmse = rmse(test_ts.values, forecast.values)

print("ARIMA MAE:", arima_mae)
print("ARIMA RMSE:", arima_rmse)

plt.figure(figsize=(10,5))

plt.plot(test_ts.index, test_ts.values, label="Actual")
plt.plot(test_ts.index, forecast.values, label="ARIMA Forecast")

plt.title("ARIMA Forecast vs Actual Demand")
plt.xlabel("Week")
plt.ylabel("Total Demand")
plt.legend()
save_fig("arima_forecast.png")
plt.show()

# 3.7. Graph Neural Network Model Development

We define the model architectures:

- **MLP** (baseline)
- **GCN** (graph convolution baseline)
- **GraphSAGE** (proposed GNN model)

## Graph Neural Network Models

Graph neural networks allow information sharing between related products
through message passing across the co-purchase network.

Three models are evaluated:

• MLP – deep learning baseline without graph structure  
• GCN – graph convolution model using normalized adjacency  
• GraphSAGE – inductive GNN using neighborhood aggregation

GraphSAGE is expected to perform best due to its robustness on large real-world graphs.

## 3.7.1. Imports and device setup

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

## 3.7.2 GNN FORECAST MODELS

In [ ]:
sample = train_data[0].to(device)

# BASELINE MODEL: MLP
# -------------------
# Purpose:
# - Provides a non-graph benchmark for forecasting accuracy.
# - Uses only historical demand (lags) per node.
# - Helps defend that any improvement is due to graph structure, not just deep learning.

class MLPForecast(nn.Module):
    def __init__(self, in_dim, hidden_dim=128, dropout=0.2):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.out = nn.Linear(hidden_dim, 1)
        self.dropout = dropout

    def forward(self, x):
        # x: (N, LAGS)
        x = F.relu(self.fc1(x))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.fc2(x))
        x = F.dropout(x, p=self.dropout, training=self.training)
        return self.out(x).squeeze(-1)  # (N,)

# GNN MODEL: GCN (Graph Convolutional Network)
# --------------------------------------------
# Purpose:
# - Aggregates neighbor information based on the product co-purchase network.
# - Captures inter-product demand dependencies (methodology goal).
#
# We first encode lag features -> hidden node embeddings, then apply GCN layers.

class GCNForecast(nn.Module):
    def __init__(self, in_dim, hidden_dim=128, dropout=0.2):
        super().__init__()
        self.lin_in = nn.Linear(in_dim, hidden_dim)
        self.gcn1 = GCNConv(hidden_dim, hidden_dim)
        self.gcn2 = GCNConv(hidden_dim, hidden_dim)
        self.out = nn.Linear(hidden_dim, 1)
        self.dropout = dropout

    def forward(self, x, edge_index, edge_weight=None):
        # x: (N, LAGS)
        h = F.relu(self.lin_in(x))
        h = F.dropout(h, p=self.dropout, training=self.training)

        h = F.relu(self.gcn1(h, edge_index, edge_weight=edge_weight))
        h = F.dropout(h, p=self.dropout, training=self.training)

        h = F.relu(self.gcn2(h, edge_index, edge_weight=edge_weight))
        h = F.dropout(h, p=self.dropout, training=self.training)

        return self.out(h).squeeze(-1)  # (N,)



# GNN MODEL: GraphSAGE
# --------------------
# Purpose:
# - Often performs well on real-world graphs, robust aggregation.
# - Learns how to combine neighbor embeddings rather than using fixed normalization.

class SAGEForecast(nn.Module):
    def __init__(self, in_dim, hidden_dim=128, dropout=0.2, aggr="mean"):
        super().__init__()

        self.lin_in = nn.Linear(in_dim, hidden_dim)

        self.sage1 = SAGEConv(
            hidden_dim,
            hidden_dim,
            aggr=aggr
        )

        self.sage2 = SAGEConv(
            hidden_dim,
            hidden_dim,
            aggr=aggr
        )

        self.out = nn.Linear(hidden_dim, 1)
        self.dropout = dropout

    def forward(self, x, edge_index):
        h = F.relu(self.lin_in(x))
        h = F.dropout(h, p=self.dropout, training=self.training)

        h = F.relu(self.sage1(h, edge_index))
        h = F.dropout(h, p=self.dropout, training=self.training)

        h = F.relu(self.sage2(h, edge_index))
        h = F.dropout(h, p=self.dropout, training=self.training)

        return self.out(h).squeeze(-1)
sage = SAGEForecast(
    in_dim=LAGS,
    hidden_dim=128,
    dropout=0.2,
    aggr="mean"
).to(device)


class SAGENoGraph(nn.Module):
    def __init__(self, in_dim, hidden_dim=128, dropout=0.2):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden_dim)
        self.fc2 = nn.Linear(hidden_dim, hidden_dim)
        self.out = nn.Linear(hidden_dim, 1)
        self.dropout = dropout

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.fc2(x))
        x = F.dropout(x, p=self.dropout, training=self.training)
        return self.out(x).squeeze(-1)


# 3.8 Model Training and Validation
**Training design:**
- Train on training snapshots
- Tune/early-stop on validation snapshots
- Report final metrics on test snapshots only (no leakage)

## Training Procedure

Models are trained using the following strategy:

• Chronological train/validation/test split  
• Mean Squared Error loss  
• Adam optimizer  
• Early stopping based on validation loss  

This prevents overfitting and ensures fair evaluation on unseen time periods.

## 3.8.1  Experimental Environment and Reproducibility

In [ ]:
# =====================================================================
# LEVEL 7.1 — DATA LOADERS & SNAPSHOT SETUP (CORRECTED & OPTIMIZED)
# =====================================================================
import random
import numpy as np
import torch
from torch_geometric.loader import NeighborLoader

# 1) Global reproducibility settings within the training loop environment
SEED = EXPERIMENT_SEED
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

# 2) Runtime device configuration
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# 3) Speed Optimization: Push the full graph memory directly to the GPU
# This shifts the neighborhood sampling execution from the CPU over to fast VRAM lanes
data = data.to(device)

# 4) Define a strict, locked generator for PyTorch Geometric neighborhood samplers
# This guarantees identical neighborhood selections are evaluated across training runs
g_loader = torch.Generator(device=device if torch.cuda.is_available() else "cpu")
g_loader.manual_seed(SEED)

# 5) Instantiating Training Graph NeighborLoader
train_loader = NeighborLoader(
    data,                     # Your graph dataset object
    num_neighbors=[15, 10],   # Your neighborhood sizing (1st and 2nd degree hops)
    batch_size=128,
    shuffle=True,
    generator=g_loader,       # Handles deterministic neighborhood selection sequence
    pin_memory=True           # Speeds up memory transfers during batch construction
)

# 6) Instantiating Testing / Evaluation Graph NeighborLoader
test_loader = NeighborLoader(
    data,
    num_neighbors=[-1],       # Pulls full neighborhood context for robust validation
    batch_size=128,
    shuffle=False,
    generator=g_loader,       # Matches the same sampler parameters
    pin_memory=True           # Speeds up evaluation steps
)

print("Section 3.8.2 executed successfully. Loaders are ready and optimized for training.")


## 3.8.2 Set Loss and helpers

In [ ]:
def mse_loss(pred, y):
    # Mean Squared Error (used for training stability)
    return torch.mean((pred - y) ** 2)

def to_device(data, device):
    return data.to(device)

## 3.8.3. Training/Evaluation loops


In [ ]:
def train_one_epoch(model, data_list, optimizer, device, use_graph=False, use_edge_weight=False):
    """
    Train for one epoch over a list of time-ordered graph snapshots.
    Each snapshot is a full-graph supervised instance (predict all node demands at once).
    """
    model.train()
    losses = []

    for data in data_list:
        data = to_device(data, device)
        optimizer.zero_grad()

        # Forward pass depends on whether model uses graph topology
        if use_graph:
            if use_edge_weight:
                pred = model(data.x, data.edge_index, data.edge_weight)
            else:
                pred = model(data.x, data.edge_index)
        else:
            pred = model(data.x)

        loss = mse_loss(pred, data.y)
        loss.backward()
        optimizer.step()

        losses.append(loss.item())

    return float(np.mean(losses))


@torch.no_grad()
def eval_one_epoch(model, data_list, device, use_graph=False, use_edge_weight=False):
    """
    Evaluate (no gradient updates) on validation or test snapshots.
    """
    model.eval()
    losses = []

    for data in data_list:
        data = to_device(data, device)

        if use_graph:
            if use_edge_weight:
                pred = model(data.x, data.edge_index, data.edge_weight)
            else:
                pred = model(data.x, data.edge_index)
        else:
            pred = model(data.x)

        loss = mse_loss(pred, data.y)
        losses.append(loss.item())

    return float(np.mean(losses))

## 3.8.4 - Early Stopping Proceedure

In [ ]:
# Cell 7.3 — Train all models with early stopping (recommended)
# -------------------------------------------------------------
# This trains MLP baseline, GCN, and GraphSAGE and keeps the best validation model.
# It ALSO records training/validation MSE history for plotting training curves.

import torch.optim as optim
import numpy as np

# =============================================================================
# 3.8.4 — EARLY STOPPING PROCEDURE
# =============================================================================

def fit_model(
    name,
    model,
    train_data,
    val_data,
    device,
    learning_rate,
    use_graph=False,
    use_edge_weight=False,
    epochs=100,
    patience=15
):
    """
    Train model using early stopping based on validation MSE.
    Learning rate is passed directly, not taken from a global variable.
    """

    print(f"[{name}] Validating data before training...")

    if train_data is None or len(train_data) == 0:
        raise ValueError(f"[{name}] train_data is empty or None")

    if val_data is None or len(val_data) == 0:
        raise ValueError(f"[{name}] val_data is empty or None")

    print(f"[{name}] Data validation passed. Starting training...")

    model = model.to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    best_val = float("inf")
    best_state = None
    bad_epochs = 0

    history = {
        "epoch": [],
        "train_mse": [],
        "val_mse": []
    }

    for ep in range(1, epochs + 1):

        train_mse = train_one_epoch(
            model,
            train_data,
            optimizer,
            device,
            use_graph=use_graph,
            use_edge_weight=use_edge_weight
        )

        val_mse = eval_one_epoch(
            model,
            val_data,
            device,
            use_graph=use_graph,
            use_edge_weight=use_edge_weight
        )

        history["epoch"].append(ep)
        history["train_mse"].append(train_mse)
        history["val_mse"].append(val_mse)

        if val_mse < best_val:
            best_val = val_mse
            best_state = {
                k: v.detach().cpu().clone()
                for k, v in model.state_dict().items()
            }
            bad_epochs = 0
        else:
            bad_epochs += 1

        if ep == 1 or ep % 10 == 0:
            print(
                f"[{name}] epoch {ep:03d} | "
                f"train MSE {train_mse:.4f} | "
                f"val MSE {val_mse:.4f} | "
                f"bad_epochs {bad_epochs}"
            )

        if bad_epochs >= patience:
            print(
                f"[{name}] Early stopping at epoch {ep} "
                f"(best val MSE {best_val:.4f})"
            )
            break

    if best_state is not None:
        model.load_state_dict(best_state)

    return model, float(best_val), history



### 3.8.4.1 - Persistent Model and Result Storage

In [ ]:
# =============================================================================
# PERSISTENT SAVE/LOAD HELPERS
# =============================================================================

def model_checkpoint_path(model_name, seed=EXPERIMENT_SEED):
    return CHECKPOINT_DIR / f"{model_name}_seed_{seed}.pt"


def history_path(model_name, seed=EXPERIMENT_SEED):
    return HISTORY_DIR / f"{model_name}_seed_{seed}_history.pkl"


def prediction_path(model_name, seed=EXPERIMENT_SEED):
    return PREDICTION_DIR / f"{model_name}_seed_{seed}_predictions.npz"


def save_trained_model(
    model,
    model_name,
    parameters,
    best_val_mse,
    seed=EXPERIMENT_SEED,
):
    path = model_checkpoint_path(model_name, seed)

    torch.save(
        {
            "model_name": model_name,
            "seed": seed,
            "model_state_dict": model.state_dict(),
            "parameters": parameters,
            "best_val_mse": float(best_val_mse),
            "lags": int(LAGS),
        },
        path,
    )

    print(f"[{model_name}] Model checkpoint saved to: {path}")


def load_trained_model(
    model_name,
    model_spec,
    device,
    seed=EXPERIMENT_SEED,
):
    path = model_checkpoint_path(model_name, seed)

    if not path.exists():
        raise FileNotFoundError(f"Checkpoint not found: {path}")

    checkpoint = torch.load(
        path,
        map_location=device,
        weights_only=False,
    )

    parameters = checkpoint["parameters"]

    model = model_spec["class"](
        in_dim=LAGS,
        hidden_dim=int(parameters["hidden_dim"]),
        dropout=float(parameters["dropout"]),
    )

    model.load_state_dict(checkpoint["model_state_dict"])
    model = model.to(device)
    model.eval()

    return (
        model,
        parameters,
        float(checkpoint["best_val_mse"]),
    )


def save_training_history(
    history,
    model_name,
    seed=EXPERIMENT_SEED,
):
    path = history_path(model_name, seed)

    with open(path, "wb") as file:
        pickle.dump(history, file)

    print(f"[{model_name}] Training history saved to: {path}")


def load_training_history(
    model_name,
    seed=EXPERIMENT_SEED,
):
    path = history_path(model_name, seed)

    if not path.exists():
        return None

    with open(path, "rb") as file:
        return pickle.load(file)


def save_model_predictions(
    model_name,
    product_true,
    product_pred,
    aggregate_true,
    aggregate_pred,
    seed=EXPERIMENT_SEED,
):
    path = prediction_path(model_name, seed)

    np.savez_compressed(
        path,
        product_true=np.asarray(product_true, dtype=np.float32),
        product_pred=np.asarray(product_pred, dtype=np.float32),
        aggregate_true=np.asarray(aggregate_true, dtype=np.float32),
        aggregate_pred=np.asarray(aggregate_pred, dtype=np.float32),
    )

    print(f"[{model_name}] Predictions saved to: {path}")


def load_model_predictions(
    model_name,
    seed=EXPERIMENT_SEED,
):
    path = prediction_path(model_name, seed)

    if not path.exists():
        raise FileNotFoundError(f"Prediction file not found: {path}")

    with np.load(path, allow_pickle=False) as saved:
        return {
            "product_true": saved["product_true"],
            "product_pred": saved["product_pred"],
            "aggregate_true": saved["aggregate_true"],
            "aggregate_pred": saved["aggregate_pred"],
        }


def all_final_checkpoints_exist(
    model_names,
    seed=EXPERIMENT_SEED,
):
    return all(
        model_checkpoint_path(name, seed).exists()
        for name in model_names
    )

## 3.8.5. FIXED hyperparameter search

In [ ]:
# =============================================================================
# 3.8.5 — FIXED HYPERPARAMETERS FOR MULTI-SEED ROBUSTNESS ANALYSIS
# =============================================================================

model_specs = {
    "MLP": {
        "class": MLPForecast,
        "use_graph": False,
        "use_edge_weight": False
    },
    "GCN": {
        "class": GCNForecast,
        "use_graph": True,
        "use_edge_weight": True
    },
    "SAGE": {
        "class": SAGEForecast,
        "use_graph": True,
        "use_edge_weight": False
    },
    "SAGE_NoGraph": {
        "class": SAGENoGraph,
        "use_graph": False,
        "use_edge_weight": False
    }
}

# Load the hyperparameters selected from the original seed-42 experiment
best_hyperparams_path = (
    CONFIG_DIR / "best_hyperparameters_seed_42.csv"
)

if not best_hyperparams_path.exists():
    raise FileNotFoundError(
        f"Seed-42 hyperparameter file not found: {best_hyperparams_path}"
    )

best_hyperparams = pd.read_csv(best_hyperparams_path)

print("=" * 80)
print("FIXED HYPERPARAMETERS FOR MULTI-SEED ANALYSIS")
print("=" * 80)
print(f"Current experiment seed: {EXPERIMENT_SEED}")
print(f"Hyperparameters loaded from: {best_hyperparams_path}")

display(best_hyperparams)

In [ ]:
# =============================================================================
# CONFIRM FIXED HYPERPARAMETERS
# =============================================================================

print(
    f"Experiment seed {EXPERIMENT_SEED} will use the "
    "fixed hyperparameters selected from the original seed-42 experiment."
)

print("No new hyperparameter configuration will be saved.")

display(best_hyperparams)

In [ ]:
print("CURRENT EXPERIMENT SEED:", EXPERIMENT_SEED)
print("FIXED HYPERPARAMETER SOURCE:", best_hyperparams_path)

for model_name in model_specs:
    print(
        model_name,
        "->",
        model_checkpoint_path(
            model_name,
            EXPERIMENT_SEED,
        )
    )

## 3.8.6 FINAL MODEL TRAINING

In [ ]:
# =============================================================================
# 3.8.6 — FINAL MODEL TRAINING WITH PERSISTENT CHECKPOINTS
# =============================================================================

FORCE_FINAL_RETRAIN = False

results = {}
trained_models = {}
histories = {}


def get_selected_params(model_name):
    row = best_hyperparams[
        best_hyperparams["Model"] == model_name
    ].iloc[0]

    return {
        "hidden_dim": int(row["Hidden Dimension"]),
        "learning_rate": float(row["Learning Rate"]),
        "dropout": float(row["Dropout"]),
    }


for model_name, spec in model_specs.items():

    params = get_selected_params(model_name)
    checkpoint_file = model_checkpoint_path(
        model_name,
        EXPERIMENT_SEED,
    )

    print("\n" + "=" * 80)
    print(f"FINAL MODEL: {model_name}")
    print("=" * 80)

    if checkpoint_file.exists() and not FORCE_FINAL_RETRAIN:

        print(
            f"[{model_name}] Existing checkpoint found. "
            "Training will be skipped."
        )

        trained_model, saved_params, best_val_mse = (
            load_trained_model(
                model_name=model_name,
                model_spec=spec,
                device=device,
                seed=EXPERIMENT_SEED,
            )
        )

        history = load_training_history(
            model_name,
            EXPERIMENT_SEED,
        )

        trained_models[model_name] = trained_model

        if history is not None:
            histories[model_name] = history

        results[model_name] = {
            "hidden_dim": saved_params["hidden_dim"],
            "learning_rate": saved_params["learning_rate"],
            "dropout": saved_params["dropout"],
            "best_val_mse": best_val_mse,
            "source": "Loaded checkpoint",
        }

    else:

        print(
            f"[{model_name}] No reusable checkpoint found. "
            "Starting final training."
        )
        print("Selected parameters:", params)

        seed_everything(EXPERIMENT_SEED)

        model = spec["class"](
            in_dim=LAGS,
            hidden_dim=params["hidden_dim"],
            dropout=params["dropout"],
        )

        trained_model, best_val_mse, history = fit_model(
            name=model_name,
            model=model,
            train_data=train_data,
            val_data=val_data,
            device=device,
            learning_rate=params["learning_rate"],
            use_graph=spec["use_graph"],
            use_edge_weight=spec["use_edge_weight"],
            epochs=100,
            patience=15,
        )

        trained_models[model_name] = trained_model
        histories[model_name] = history

        results[model_name] = {
            "hidden_dim": params["hidden_dim"],
            "learning_rate": params["learning_rate"],
            "dropout": params["dropout"],
            "best_val_mse": best_val_mse,
            "source": "New training",
        }

        # Save immediately after each model finishes.
        save_trained_model(
            model=trained_model,
            model_name=model_name,
            parameters=params,
            best_val_mse=best_val_mse,
            seed=EXPERIMENT_SEED,
        )

        save_training_history(
            history=history,
            model_name=model_name,
            seed=EXPERIMENT_SEED,
        )

        # Release unused cached GPU memory.
        if torch.cuda.is_available():
            torch.cuda.empty_cache()


final_training_summary = (
    pd.DataFrame(results)
    .T
    .reset_index()
    .rename(columns={"index": "Model"})
)

final_training_summary_path = (
    METRICS_DIR
    / f"final_training_summary_seed_{EXPERIMENT_SEED}.csv"
)

final_training_summary.to_csv(
    final_training_summary_path,
    index=False,
)

display(final_training_summary)

print(
    "Final training summary saved to:",
    final_training_summary_path,
)

# 3.9. Evaluation Metrics and Performance
We report:
- MAE, RMSE, MAPE, R² (product-level)
- Total-demand evaluation (to compare fairly with ARIMA)

## 3.9.1 Metric Functions

In [ ]:
@torch.no_grad()
def collect_preds(model, data_list, device, use_graph=False, use_edge_weight=False):
    """
    Collect product-level predictions and targets across all snapshots in a split.
    Returns flattened arrays over (time, nodes).
    """
    model.eval()
    preds_all = []
    y_all = []

    for data in data_list:
        data = data.to(device)

        if use_graph:
            if use_edge_weight:
                pred = model(data.x, data.edge_index, data.edge_weight)
            else:
                pred = model(data.x, data.edge_index)
        else:
            pred = model(data.x)

        preds_all.append(pred.detach().cpu().numpy().ravel())
        y_all.append(data.y.detach().cpu().numpy().ravel())

    preds_all = np.concatenate(preds_all, axis=0)
    y_all = np.concatenate(y_all, axis=0)
    return preds_all, y_all


@torch.no_grad()
def collect_total_preds(model, data_list, device, use_graph=False, use_edge_weight=False):
    """
    Collect aggregated (TOTAL) weekly demand predictions and targets across snapshots.
    This enables fair comparison against ARIMA (which predicts a single total per week).
    Returns arrays of length = number of snapshots.
    """
    model.eval()
    preds_total = []
    y_total = []

    for data in data_list:
        data = data.to(device)

        if use_graph:
            if use_edge_weight:
                pred = model(data.x, data.edge_index, data.edge_weight)
            else:
                pred = model(data.x, data.edge_index)
        else:
            pred = model(data.x)

        preds_total.append(float(pred.sum().detach().cpu().item()))
        y_total.append(float(data.y.sum().detach().cpu().item()))

    return np.array(preds_total), np.array(y_total)


def mae(y_true, y_pred):
    return float(np.mean(np.abs(y_true - y_pred)))

def rmse(y_true, y_pred):
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))

def mape(y_true, y_pred, eps=1e-6, min_demand=1.0):
    """
    MAPE can explode when y_true is near 0.
    We compute MAPE only where y_true >= min_demand (defensible).
    """
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    mask = y_true >= min_demand
    if mask.sum() == 0:
        return np.nan
    return float(np.mean(np.abs((y_true[mask] - y_pred[mask]) / (y_true[mask] + eps))) * 100.0)

def r2(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    ss_res = np.sum((y_true - y_pred) ** 2)
    ss_tot = np.sum((y_true - np.mean(y_true)) ** 2)
    return float(1 - (ss_res / (ss_tot + 1e-12)))

## 3.9.2. EVALUATION

### 3.9.2.1 — Product-level evaluation (MLP/GCN/SAGE)

In [ ]:
# =============================================================================
# 3.9.2.1 — PRODUCT-LEVEL EVALUATION WITH SAVED PREDICTIONS
# =============================================================================

comparison = []

model_cfg = {
    "MLP": dict(
        use_graph=False,
        use_edge_weight=False,
    ),
    "GCN": dict(
        use_graph=True,
        use_edge_weight=True,
    ),
    "SAGE": dict(
        use_graph=True,
        use_edge_weight=False,
    ),
    "SAGE_NoGraph": dict(
        use_graph=False,
        use_edge_weight=False,
    ),
}

prediction_cache = {}

for name, model in trained_models.items():

    if name not in model_cfg:
        print(
            f"Skipping {name}: "
            "no model configuration found."
        )
        continue

    saved_prediction_file = prediction_path(
        name,
        EXPERIMENT_SEED,
    )

    if saved_prediction_file.exists():

        print(
            f"[{name}] Loading saved test predictions."
        )

        saved = load_model_predictions(
            name,
            EXPERIMENT_SEED,
        )

        y_true = saved["product_true"]
        y_pred = saved["product_pred"]
        true_tot = saved["aggregate_true"]
        pred_tot = saved["aggregate_pred"]

    else:

        print(
            f"[{name}] Generating test predictions."
        )

        cfg = model_cfg[name]

        y_pred, y_true = collect_preds(
            model,
            test_data,
            device,
            **cfg,
        )

        pred_tot, true_tot = collect_total_preds(
            model,
            test_data,
            device,
            **cfg,
        )

        save_model_predictions(
            model_name=name,
            product_true=y_true,
            product_pred=y_pred,
            aggregate_true=true_tot,
            aggregate_pred=pred_tot,
            seed=EXPERIMENT_SEED,
        )

    prediction_cache[name] = {
        "product_true": np.asarray(y_true),
        "product_pred": np.asarray(y_pred),
        "aggregate_true": np.asarray(true_tot),
        "aggregate_pred": np.asarray(pred_tot),
    }

    comparison.append({
        "model": name,
        "MAE": mae(y_true, y_pred),
        "RMSE": rmse(y_true, y_pred),
        "MAPE(>=1)": mape(
            y_true,
            y_pred,
            min_demand=1.0,
        ),
        "R2": r2(y_true, y_pred),
    })


comparison_df = (
    pd.DataFrame(comparison)
    .sort_values("RMSE")
    .reset_index(drop=True)
)

comparison_path = (
    METRICS_DIR
    / f"product_metrics_seed_{EXPERIMENT_SEED}.csv"
)

comparison_df.to_csv(
    comparison_path,
    index=False,
)

display(comparison_df)

print(
    "Product-level metrics saved to:",
    comparison_path,
)

### Cell 3.9.2.2 — Total-demand evaluation

In [ ]:
# =============================================================================
# 3.9.2.2 — TOTAL-DEMAND EVALUATION FROM SAVED PREDICTIONS
# =============================================================================

total_comparison = []

for name in model_cfg:

    if name in prediction_cache:
        saved = prediction_cache[name]
    else:
        saved = load_model_predictions(
            name,
            EXPERIMENT_SEED,
        )

    true_tot = np.asarray(
        saved["aggregate_true"],
        dtype=float,
    )

    pred_tot = np.asarray(
        saved["aggregate_pred"],
        dtype=float,
    )

    total_comparison.append({
        "model": name,
        "MAE_total": mae(true_tot, pred_tot),
        "RMSE_total": rmse(true_tot, pred_tot),
        "MAPE_total(>=1)": mape(
            true_tot,
            pred_tot,
            min_demand=1.0,
        ),
        "R2_total": r2(true_tot, pred_tot),
    })


total_df = pd.DataFrame(total_comparison)

# Add ARIMA results.
if "test_ts" in globals() and "forecast" in globals():

    arima_true = np.asarray(
        test_ts.values,
        dtype=float,
    )

    arima_pred = np.asarray(
        forecast.values,
        dtype=float,
    )

    arima_prediction_path = (
        PREDICTION_DIR
        / f"ARIMA_seed_{EXPERIMENT_SEED}_predictions.npz"
    )

    np.savez_compressed(
        arima_prediction_path,
        aggregate_true=arima_true.astype(np.float32),
        aggregate_pred=arima_pred.astype(np.float32),
    )

    total_df = pd.concat(
        [
            total_df,
            pd.DataFrame([
                {
                    "model": "ARIMA",
                    "MAE_total": mae(
                        arima_true,
                        arima_pred,
                    ),
                    "RMSE_total": rmse(
                        arima_true,
                        arima_pred,
                    ),
                    "MAPE_total(>=1)": mape(
                        arima_true,
                        arima_pred,
                        min_demand=1.0,
                    ),
                    "R2_total": r2(
                        arima_true,
                        arima_pred,
                    ),
                }
            ]),
        ],
        ignore_index=True,
    )

else:
    print(
        "ARIMA variables are unavailable. "
        "Run the ARIMA section once to save its predictions."
    )


total_df = (
    total_df
    .sort_values("RMSE_total")
    .reset_index(drop=True)
)

total_metrics_path = (
    METRICS_DIR
    / f"aggregate_metrics_seed_{EXPERIMENT_SEED}.csv"
)

total_df.to_csv(
    total_metrics_path,
    index=False,
)

display(total_df)

print(
    "Aggregate metrics saved to:",
    total_metrics_path,
)

### 3.9.2.3 Performance Comparison Summary

In [ ]:
# =============================================================================
# FINAL PERFORMANCE SUMMARY
# =============================================================================

from IPython.display import display

# ----------------------------
# PRODUCT-LEVEL RESULTS
# ----------------------------
product_df = comparison_df.copy()
product_df.insert(0, "Rank", range(1, len(product_df)+1))

print("="*80)
print("TABLE 4.X: PRODUCT-LEVEL DEMAND FORECASTING PERFORMANCE")
print("="*80)

display(product_df.style
        .format({
            "MAE":"{:.4f}",
            "RMSE":"{:.4f}",
            "MAPE(>=1)":"{:.2f}",
            "R2":"{:.4f}"
        })
        .highlight_min(subset=["MAE","RMSE","MAPE(>=1)"], color="lightgreen")
        .highlight_max(subset=["R2"], color="lightgreen")
)

best_prod = product_df.iloc[0]

print(f"\n🏆 Best Product-Level Model : {best_prod['model']}")
print(f"RMSE : {best_prod['RMSE']:.4f}")
print(f"MAE  : {best_prod['MAE']:.4f}")
print(f"MAPE : {best_prod['MAPE(>=1)']:.2f}%")
print(f"R²   : {best_prod['R2']:.4f}")


# ----------------------------
# PRODUCT-LEVEL BAR CHART
# ----------------------------
ax = comparison_df.set_index("model")[["RMSE","MAE"]].plot(
    kind="bar",
    figsize=(8,5)
)

plt.title("Product-Level Forecasting Performance")
plt.ylabel("Error")
plt.xlabel("")
plt.xticks(rotation=0)
plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()
plt.show()



# =============================================================================
# TOTAL DEMAND RESULTS
# =============================================================================

total_results = total_df.copy()
total_results.insert(0, "Rank", range(1, len(total_results)+1))

print("\n")
print("="*80)
print("TABLE 4.X: TOTAL DEMAND FORECASTING PERFORMANCE")
print("="*80)

display(total_results.style
        .format({
            "MAE_total":"{:.2f}",
            "RMSE_total":"{:.2f}",
            "MAPE_total(>=1)":"{:.2f}",
            "R2_total":"{:.4f}"
        })
        .highlight_min(subset=["MAE_total","RMSE_total","MAPE_total(>=1)"], color="lightgreen")
        .highlight_max(subset=["R2_total"], color="lightgreen")
)

best_total = total_results.iloc[0]

print(f"\n🏆 Best Total-Demand Model : {best_total['model']}")
print(f"RMSE : {best_total['RMSE_total']:.2f}")
print(f"MAE  : {best_total['MAE_total']:.2f}")
print(f"MAPE : {best_total['MAPE_total(>=1)']:.2f}%")
print(f"R²   : {best_total['R2_total']:.4f}")


# ----------------------------
# TOTAL DEMAND BAR CHART
# ----------------------------
ax = total_df.set_index("model")[["RMSE_total","MAE_total"]].plot(
    kind="bar",
    figsize=(8,5)
)

plt.title("Total Demand Forecasting Performance")
plt.ylabel("Error")
plt.xlabel("")
plt.xticks(rotation=0)
plt.grid(axis="y", linestyle="--", alpha=0.5)
plt.tight_layout()

save_fig("performance_comparison.png")

plt.show()

In [ ]:
## RUN PLATFORM REPORTING

import platform
import sys
import torch_geometric

print("Python:", sys.version)
print("Platform:", platform.platform())
print("PyTorch:", torch.__version__)
print("PyG:", torch_geometric.__version__)
print("CUDA:", torch.version.cuda)
print("GPU:", torch.cuda.get_device_name(0))

## 3.9.3 — PERFORMANCE IMPROVEMENT AGAINST BASELINE MODELS

In [ ]:
def improvement_pct(baseline, model):
    """
    Percentage improvement where lower error is better.
    Positive value means model improves over baseline.
    """
    return ((baseline - model) / baseline) * 100


# -----------------------------
# Product-level improvement
# -----------------------------
product_improvement_rows = []

baseline_models_product = ["MLP", "SAGE_NoGraph"]
gnn_models_product = ["SAGE", "GCN"]

for gnn_model in gnn_models_product:
    for baseline_model in baseline_models_product:

        gnn_row = comparison_df[comparison_df["model"] == gnn_model].iloc[0]
        base_row = comparison_df[comparison_df["model"] == baseline_model].iloc[0]

        product_improvement_rows.append({
            "GNN Model": gnn_model,
            "Baseline": baseline_model,
            "RMSE Improvement (%)": improvement_pct(base_row["RMSE"], gnn_row["RMSE"]),
            "MAE Improvement (%)": improvement_pct(base_row["MAE"], gnn_row["MAE"]),
            "Meets 3% RMSE Target": improvement_pct(base_row["RMSE"], gnn_row["RMSE"]) >= 3,
            "Meets 3% MAE Target": improvement_pct(base_row["MAE"], gnn_row["MAE"]) >= 3,
        })

product_improvement_df = pd.DataFrame(product_improvement_rows)

print("="*80)
print("PRODUCT-LEVEL 3% PERFORMANCE IMPROVEMENT ANALYSIS")
print("="*80)

display(
    product_improvement_df.style.format({
        "RMSE Improvement (%)": "{:.2f}",
        "MAE Improvement (%)": "{:.2f}",
    })
)


# -----------------------------
# Total-demand improvement
# -----------------------------
total_improvement_rows = []

baseline_models_total = ["MLP", "SAGE_NoGraph", "ARIMA"]
gnn_models_total = ["SAGE", "GCN"]

for gnn_model in gnn_models_total:
    for baseline_model in baseline_models_total:

        gnn_row = total_df[total_df["model"] == gnn_model].iloc[0]
        base_row = total_df[total_df["model"] == baseline_model].iloc[0]

        total_improvement_rows.append({
            "GNN Model": gnn_model,
            "Baseline": baseline_model,
            "RMSE Improvement (%)": improvement_pct(base_row["RMSE_total"], gnn_row["RMSE_total"]),
            "MAE Improvement (%)": improvement_pct(base_row["MAE_total"], gnn_row["MAE_total"]),
            "Meets 3% RMSE Target": improvement_pct(base_row["RMSE_total"], gnn_row["RMSE_total"]) >= 3,
            "Meets 3% MAE Target": improvement_pct(base_row["MAE_total"], gnn_row["MAE_total"]) >= 3,
        })

total_improvement_df = pd.DataFrame(total_improvement_rows)

print("\n" + "="*80)
print("TOTAL-DEMAND 3% PERFORMANCE IMPROVEMENT ANALYSIS")
print("="*80)

display(
    total_improvement_df.style.format({
        "RMSE Improvement (%)": "{:.2f}",
        "MAE Improvement (%)": "{:.2f}",
    })
)

## 3.9.4 DIEBOLD-MARIANO SIGNIFICANCE TEST

In [ ]:
import numpy as np
import pandas as pd

from scipy.stats import t
from statsmodels.stats.multitest import multipletests


def diebold_mariano(errors_model1, errors_model2, alternative="two_sided"):
    """
    Compare the predictive accuracy of two forecasting models using
    squared-error loss.

    Parameters
    ----------
    errors_model1 : array-like
        Forecast errors from the first model.
    errors_model2 : array-like
        Forecast errors from the second model.
    alternative : {"two_sided", "less", "greater"}
        Alternative hypothesis for the test.

    Returns
    -------
    dm_stat : float
        Diebold–Mariano test statistic.
    p_value : float
        Corresponding p-value.

    Interpretation
    --------------
    A negative DM statistic means that model 1 has a lower average
    squared forecast error than model 2.

    A positive DM statistic means that model 1 has a higher average
    squared forecast error than model 2.
    """

    e1 = np.asarray(errors_model1, dtype=float).reshape(-1)
    e2 = np.asarray(errors_model2, dtype=float).reshape(-1)

    if e1.shape != e2.shape:
        raise ValueError(
            "The forecast-error arrays must have the same shape."
        )

    if len(e1) < 2:
        raise ValueError(
            "At least two forecast errors are required for the DM test."
        )

    if not np.all(np.isfinite(e1)) or not np.all(np.isfinite(e2)):
        raise ValueError(
            "The forecast-error arrays contain NaN or infinite values."
        )

    # Squared-error loss differential:
    # d_t = L(e1_t) - L(e2_t)
    d = (e1 ** 2) - (e2 ** 2)

    mean_d = np.mean(d)
    var_d = np.var(d, ddof=1)

    if np.isclose(var_d, 0):
        return np.nan, np.nan

    dm_stat = mean_d / np.sqrt(var_d / len(d))
    degrees_of_freedom = len(d) - 1

    if alternative == "two_sided":
        p_value = 2 * (
            1 - t.cdf(abs(dm_stat), df=degrees_of_freedom)
        )

    elif alternative == "less":
        p_value = t.cdf(
            dm_stat,
            df=degrees_of_freedom
        )

    elif alternative == "greater":
        p_value = 1 - t.cdf(
            dm_stat,
            df=degrees_of_freedom
        )

    else:
        raise ValueError(
            "alternative must be 'two_sided', 'less', or 'greater'."
        )

    return float(dm_stat), float(p_value)

In [ ]:
# =============================================================================
# PRODUCT-LEVEL FORECAST ERRORS
# =============================================================================

prediction_errors = {}

product_model_configurations = {
    "MLP": {
        "use_graph": False,
        "use_edge_weight": False
    },

    "GCN": {
        "use_graph": True,
        "use_edge_weight": True
    },

    "SAGE": {
        "use_graph": True,
        "use_edge_weight": False
    },

    "SAGE_NoGraph": {
        "use_graph": False,
        "use_edge_weight": False
    }
}


for model_name, configuration in product_model_configurations.items():

    predictions, actual_values = collect_preds(
        trained_models[model_name],
        test_data,
        device,
        **configuration
    )

    predictions = np.asarray(predictions, dtype=float).reshape(-1)
    actual_values = np.asarray(actual_values, dtype=float).reshape(-1)

    if predictions.shape != actual_values.shape:
        raise ValueError(
            f"Prediction and actual-value shapes do not match "
            f"for {model_name}: "
            f"{predictions.shape} versus {actual_values.shape}"
        )

    prediction_errors[model_name] = (
        actual_values - predictions
    )


print("Product-level forecast errors generated successfully.")

for model_name, errors in prediction_errors.items():
    print(
        f"{model_name}: {len(errors):,} forecast errors"
    )

In [ ]:
# =============================================================================
# PRODUCT-LEVEL DIEBOLD–MARIANO TEST
# =============================================================================

product_comparisons = [
    ("SAGE", "MLP"),
    ("SAGE", "GCN"),
    ("SAGE", "SAGE_NoGraph")
]

product_dm_records = []


for model1, model2 in product_comparisons:

    dm_statistic, p_value = diebold_mariano(
        prediction_errors[model1],
        prediction_errors[model2],
        alternative="two_sided"
    )

    product_dm_records.append({
        "Comparison": f"{model1} vs {model2}",
        "DM Statistic": dm_statistic,
        "p-value": p_value,
        "Significant (5%)": (
            p_value < 0.05
            if np.isfinite(p_value)
            else False
        )
    })


dm_results = pd.DataFrame(product_dm_records)


print("=" * 80)
print("DIEBOLD–MARIANO TEST RESULTS — PRODUCT-LEVEL FORECASTING")
print("=" * 80)

display(
    dm_results.style.format({
        "DM Statistic": "{:.4f}",
        "p-value": "{:.6f}"
    })
)

In [ ]:
# =============================================================================
# HOLM CORRECTION — PRODUCT-LEVEL DM TESTS
# =============================================================================

def add_holm_correction(results_df, alpha=0.05):
    """
    Apply the Holm–Bonferroni correction to a collection of p-values.
    """

    corrected_results = results_df.copy()

    valid_mask = corrected_results["p-value"].notna()

    corrected_results["Holm-adjusted p-value"] = np.nan
    corrected_results["Significant after Holm correction"] = False

    if valid_mask.any():

        reject_null, adjusted_p_values, _, _ = multipletests(
            corrected_results.loc[
                valid_mask, "p-value"
            ].to_numpy(),
            alpha=alpha,
            method="holm"
        )

        corrected_results.loc[
            valid_mask,
            "Holm-adjusted p-value"
        ] = adjusted_p_values

        corrected_results.loc[
            valid_mask,
            "Significant after Holm correction"
        ] = reject_null

    return corrected_results


dm_results_holm = add_holm_correction(
    dm_results,
    alpha=0.05
)


print("=" * 80)
print("PRODUCT-LEVEL DM TEST RESULTS — HOLM-ADJUSTED")
print("=" * 80)

display(
    dm_results_holm.style.format({
        "DM Statistic": "{:.4f}",
        "p-value": "{:.6f}",
        "Holm-adjusted p-value": "{:.6f}"
    })
)

In [ ]:
# =============================================================================
# AGGREGATE-DEMAND FORECAST ERRORS
# =============================================================================

total_prediction_errors = {}


aggregate_model_configurations = {
    "MLP": {
        "use_graph": False,
        "use_edge_weight": False
    },

    "GCN": {
        "use_graph": True,
        "use_edge_weight": True
    },

    "SAGE": {
        "use_graph": True,
        "use_edge_weight": False
    },

    "SAGE_NoGraph": {
        "use_graph": False,
        "use_edge_weight": False
    }
}


for model_name, configuration in aggregate_model_configurations.items():

    predictions, actual_values = collect_preds(
        trained_models[model_name],
        test_data,
        device,
        **configuration
    )

    predictions = np.asarray(predictions, dtype=float).reshape(
        len(test_data),
        -1
    )

    actual_values = np.asarray(actual_values, dtype=float).reshape(
        len(test_data),
        -1
    )

    # Sum product-level forecasts and actual demand within each week
    predicted_weekly_totals = predictions.sum(axis=1)
    actual_weekly_totals = actual_values.sum(axis=1)

    total_prediction_errors[model_name] = (
        actual_weekly_totals - predicted_weekly_totals
    )


# Convert ARIMA actual values and forecasts to one-dimensional arrays
arima_actual_values = np.asarray(
    test_ts.values,
    dtype=float
).reshape(-1)

arima_forecasts = np.asarray(
    forecast.values,
    dtype=float
).reshape(-1)


# Align all aggregate forecasts to a common test-period length
neural_error_length = min(
    len(errors)
    for errors in total_prediction_errors.values()
)

common_length = min(
    neural_error_length,
    len(arima_actual_values),
    len(arima_forecasts)
)


if common_length < 2:
    raise ValueError(
        "Insufficient overlapping observations for aggregate DM testing."
    )


# Retain the final common test period for every neural model
for model_name in total_prediction_errors:

    total_prediction_errors[model_name] = (
        np.asarray(
            total_prediction_errors[model_name],
            dtype=float
        ).reshape(-1)[-common_length:]
    )


# Add ARIMA aggregate forecast errors
total_prediction_errors["ARIMA"] = (
    arima_actual_values[-common_length:]
    - arima_forecasts[-common_length:]
)


print(
    "Aggregate-demand forecast errors generated and aligned successfully."
)

for model_name, errors in total_prediction_errors.items():
    print(
        f"{model_name}: {len(errors)} weekly forecast errors"
    )

In [ ]:
# =============================================================================
# AGGREGATE-DEMAND DIEBOLD–MARIANO TEST
# =============================================================================

total_comparisons = [
    ("SAGE", "MLP"),
    ("SAGE", "GCN"),
    ("SAGE", "SAGE_NoGraph"),
    ("SAGE", "ARIMA")
]

total_dm_records = []


for model1, model2 in total_comparisons:

    dm_statistic, p_value = diebold_mariano(
        total_prediction_errors[model1],
        total_prediction_errors[model2],
        alternative="two_sided"
    )

    total_dm_records.append({
        "Comparison": f"{model1} vs {model2}",
        "DM Statistic": dm_statistic,
        "p-value": p_value,
        "Significant (5%)": (
            p_value < 0.05
            if np.isfinite(p_value)
            else False
        )
    })


total_dm_results = pd.DataFrame(total_dm_records)


print("=" * 80)
print("DIEBOLD–MARIANO TEST RESULTS — AGGREGATE DEMAND")
print("=" * 80)

display(
    total_dm_results.style.format({
        "DM Statistic": "{:.4f}",
        "p-value": "{:.6f}"
    })
)

In [ ]:
# =============================================================================
# HOLM CORRECTION — AGGREGATE-DEMAND DM TESTS
# =============================================================================

total_dm_results_holm = add_holm_correction(
    total_dm_results,
    alpha=0.05
)


print("=" * 80)
print("AGGREGATE-DEMAND DM TEST RESULTS — HOLM-ADJUSTED")
print("=" * 80)

display(
    total_dm_results_holm.style.format({
        "DM Statistic": "{:.4f}",
        "p-value": "{:.6f}",
        "Holm-adjusted p-value": "{:.6f}"
    })
)

## 3.9.5 TRAINING DYNAMICS/LOSS CURVE

In [ ]:
for model_name, history in histories.items():

    plt.figure(figsize=(7, 5))

    plt.plot(history["epoch"], history["train_mse"], label="Training MSE")
    plt.plot(history["epoch"], history["val_mse"], label="Validation MSE")

    plt.xlabel("Epoch")
    plt.ylabel("MSE")
    plt.title(f"{model_name} Training and Validation Curve")
    plt.legend()
    plt.grid(True)

    save_fig(f"{model_name.lower()}_training_curve.png")

    plt.show()

4.0 Bootstrap Confidence Intervals for Forecasting Accuracy

This section estimates 95% non-parametric bootstrap confidence intervals for RMSE and MAE. Resampling is performed at the **weekly snapshot level**, so that all product observations within a selected week remain together. This preserves the within-week dependence among product forecasts more appropriately than independently resampling individual product observations. A fixed bootstrap seed is used for reproducibility.

In [ ]:
# =============================================================================
# BOOTSTRAP UTILITIES — WEEK/SNAPSHOT-LEVEL RESAMPLING
# =============================================================================

BOOTSTRAP_SAMPLES = 1_000
BOOTSTRAP_ALPHA = 0.05
BOOTSTRAP_SEED = 42


def _validate_bootstrap_arrays(y_true, y_pred):
    """Validate and return finite NumPy arrays with identical shapes."""
    y_true = np.asarray(y_true, dtype=float)
    y_pred = np.asarray(y_pred, dtype=float)

    if y_true.shape != y_pred.shape:
        raise ValueError(
            f"y_true and y_pred must have identical shapes; "
            f"received {y_true.shape} and {y_pred.shape}."
        )

    if y_true.ndim != 2:
        raise ValueError(
            "Expected two-dimensional arrays shaped "
            "(number_of_weeks, observations_per_week)."
        )

    if y_true.shape[0] < 2:
        raise ValueError("At least two weekly snapshots are required.")

    if not (np.isfinite(y_true).all() and np.isfinite(y_pred).all()):
        raise ValueError("Bootstrap inputs contain NaN or infinite values.")

    return y_true, y_pred


def weekly_cluster_bootstrap_ci(
    y_true,
    y_pred,
    n_bootstrap=BOOTSTRAP_SAMPLES,
    alpha=BOOTSTRAP_ALPHA,
    random_state=BOOTSTRAP_SEED,
):
    """
    Estimate percentile bootstrap confidence intervals for RMSE and MAE.

    Weekly snapshots are sampled with replacement. For product-level results,
    every product observation within a sampled week is retained as a cluster.
    For aggregate-demand results, each week contains one observation.
    """
    y_true, y_pred = _validate_bootstrap_arrays(y_true, y_pred)
    rng = np.random.default_rng(random_state)
    n_weeks = y_true.shape[0]

    rmse_samples = np.empty(n_bootstrap, dtype=float)
    mae_samples = np.empty(n_bootstrap, dtype=float)

    for bootstrap_index in range(n_bootstrap):
        sampled_weeks = rng.integers(0, n_weeks, size=n_weeks)
        sampled_true = y_true[sampled_weeks].reshape(-1)
        sampled_pred = y_pred[sampled_weeks].reshape(-1)

        residuals = sampled_true - sampled_pred
        rmse_samples[bootstrap_index] = np.sqrt(np.mean(residuals ** 2))
        mae_samples[bootstrap_index] = np.mean(np.abs(residuals))

    lower_percentile = 100 * (alpha / 2)
    upper_percentile = 100 * (1 - alpha / 2)

    original_true = y_true.reshape(-1)
    original_pred = y_pred.reshape(-1)
    original_residuals = original_true - original_pred

    return {
        "RMSE": float(np.sqrt(np.mean(original_residuals ** 2))),
        "RMSE CI Lower": float(np.percentile(rmse_samples, lower_percentile)),
        "RMSE CI Upper": float(np.percentile(rmse_samples, upper_percentile)),
        "MAE": float(np.mean(np.abs(original_residuals))),
        "MAE CI Lower": float(np.percentile(mae_samples, lower_percentile)),
        "MAE CI Upper": float(np.percentile(mae_samples, upper_percentile)),
    }


def format_bootstrap_table(results_df):
    """Add dissertation-ready confidence-interval display columns."""
    formatted = results_df.copy()
    formatted["RMSE (95% CI)"] = formatted.apply(
        lambda row: (
            f'{row["RMSE"]:.4f} '
            f'({row["RMSE CI Lower"]:.4f}–{row["RMSE CI Upper"]:.4f})'
        ),
        axis=1,
    )
    formatted["MAE (95% CI)"] = formatted.apply(
        lambda row: (
            f'{row["MAE"]:.4f} '
            f'({row["MAE CI Lower"]:.4f}–{row["MAE CI Upper"]:.4f})'
        ),
        axis=1,
    )
    return formatted[["Model", "RMSE (95% CI)", "MAE (95% CI)"]]


print(
    f"Bootstrap configuration: {BOOTSTRAP_SAMPLES:,} resamples, "
    f"{int((1 - BOOTSTRAP_ALPHA) * 100)}% percentile CI, "
    f"seed = {BOOTSTRAP_SEED}."
)


In [ ]:
# =============================================================================
# PRODUCT-LEVEL BOOTSTRAP CONFIDENCE INTERVALS
# =============================================================================

bootstrap_model_configurations = {
    "MLP": dict(use_graph=False, use_edge_weight=False),
    "GCN": dict(use_graph=True, use_edge_weight=True),
    "SAGE": dict(use_graph=True, use_edge_weight=False),
    "SAGE_NoGraph": dict(use_graph=False, use_edge_weight=False),
}

product_bootstrap_records = []

for model_name in bootstrap_model_configurations:

    saved = load_model_predictions(
        model_name,
        EXPERIMENT_SEED,
    )

    predictions = np.asarray(
        saved["product_pred"],
        dtype=float,
    ).reshape(len(test_data), -1)

    actual_values = np.asarray(
        saved["product_true"],
        dtype=float,
    ).reshape(len(test_data), -1)

    bootstrap_statistics = weekly_cluster_bootstrap_ci(
        y_true=actual_values,
        y_pred=predictions,
        n_bootstrap=BOOTSTRAP_SAMPLES,
        alpha=BOOTSTRAP_ALPHA,
        random_state=BOOTSTRAP_SEED,
    )

    product_bootstrap_records.append({
        "Model": (
            "GraphSAGE"
            if model_name == "SAGE"
            else model_name
        ),
        **bootstrap_statistics,
    })

product_bootstrap_results = (
    pd.DataFrame(product_bootstrap_records)
      .sort_values("RMSE")
      .reset_index(drop=True)
)

product_bootstrap_table = format_bootstrap_table(product_bootstrap_results)

print("=" * 90)
print("PRODUCT-LEVEL RMSE AND MAE — 95% BOOTSTRAP CONFIDENCE INTERVALS")
print("=" * 90)
display(product_bootstrap_table)

# Optional export for direct use in Chapter 4.
product_bootstrap_output = (
    ROBUSTNESS_DIR
    / f"product_bootstrap_seed_{EXPERIMENT_SEED}.csv"
)

product_bootstrap_table.to_csv(
    product_bootstrap_output,
    index=False,
)

print(
    "Product bootstrap results saved to:",
    product_bootstrap_output,
)

In [ ]:
# =============================================================================
# AGGREGATE-DEMAND BOOTSTRAP CONFIDENCE INTERVALS
# =============================================================================

aggregate_predictions = {}
aggregate_actual_values = None

for model_name in bootstrap_model_configurations:

    saved = load_model_predictions(
        model_name,
        EXPERIMENT_SEED,
    )

    current_predictions = np.asarray(
        saved["aggregate_pred"],
        dtype=float,
    ).reshape(-1)

    current_actuals = np.asarray(
        saved["aggregate_true"],
        dtype=float,
    ).reshape(-1)

    aggregate_predictions[model_name] = (
        current_predictions
    )

    if aggregate_actual_values is None:
        aggregate_actual_values = current_actuals

    elif not np.allclose(
        aggregate_actual_values,
        current_actuals,
    ):
        raise ValueError(
            "Aggregate actual values differ "
            "across saved model predictions."
        )

# ARIMA operates only at the aggregate-demand level.
arima_prediction_path = (
    PREDICTION_DIR
    / f"ARIMA_seed_{EXPERIMENT_SEED}_predictions.npz"
)

if not arima_prediction_path.exists():
    raise FileNotFoundError(
        "Saved ARIMA predictions were not found. "
        "Run the aggregate evaluation cell once."
    )

with np.load(
    arima_prediction_path,
    allow_pickle=False,
) as arima_saved:

    arima_actual_values = np.asarray(
        arima_saved["aggregate_true"],
        dtype=float,
    ).reshape(-1)

    arima_predictions = np.asarray(
        arima_saved["aggregate_pred"],
        dtype=float,
    ).reshape(-1)

common_length = min(
    len(aggregate_actual_values),
    len(arima_actual_values),
    len(arima_predictions),
    *(len(values) for values in aggregate_predictions.values()),
)

if common_length < 2:
    raise ValueError("Insufficient aligned weeks for aggregate bootstrap analysis.")

# Use the same final common evaluation period for every model.
aligned_actual_values = aggregate_actual_values[-common_length:]
aggregate_predictions = {
    model_name: values[-common_length:]
    for model_name, values in aggregate_predictions.items()
}
aggregate_predictions["ARIMA"] = arima_predictions[-common_length:]

# Confirm that the neural and ARIMA actual series refer to the same evaluation period.
aligned_arima_actuals = arima_actual_values[-common_length:]
if not np.allclose(aligned_actual_values, aligned_arima_actuals):
    print(
        "Warning: neural-model and ARIMA actual totals are not numerically "
        "identical. The neural-model actual series is used for neural models, "
        "and the ARIMA actual series is used for ARIMA."
    )

aggregate_bootstrap_records = []

for model_name, predictions in aggregate_predictions.items():
    model_actuals = (
        aligned_arima_actuals if model_name == "ARIMA" else aligned_actual_values
    )

    bootstrap_statistics = weekly_cluster_bootstrap_ci(
        y_true=model_actuals.reshape(-1, 1),
        y_pred=np.asarray(predictions, dtype=float).reshape(-1, 1),
        n_bootstrap=BOOTSTRAP_SAMPLES,
        alpha=BOOTSTRAP_ALPHA,
        random_state=BOOTSTRAP_SEED,
    )

    aggregate_bootstrap_records.append({
        "Model": "GraphSAGE" if model_name == "SAGE" else model_name,
        **bootstrap_statistics,
    })

aggregate_bootstrap_results = (
    pd.DataFrame(aggregate_bootstrap_records)
      .sort_values("RMSE")
      .reset_index(drop=True)
)

aggregate_bootstrap_table = format_bootstrap_table(aggregate_bootstrap_results)

print("=" * 90)
print("AGGREGATE-DEMAND RMSE AND MAE — 95% BOOTSTRAP CONFIDENCE INTERVALS")
print("=" * 90)
display(aggregate_bootstrap_table)

# Optional export for direct use in Chapter 4.
aggregate_bootstrap_output = (
    ROBUSTNESS_DIR
    / f"aggregate_bootstrap_seed_{EXPERIMENT_SEED}.csv"
)

aggregate_bootstrap_table.to_csv(
    aggregate_bootstrap_output,
    index=False,
)

print(
    "Aggregate bootstrap results saved to:",
    aggregate_bootstrap_output,
)


# 5.0 - ## Multi-Seed Variability Analysis

In [ ]:
# =============================================================================
# MULTI-SEED VARIABILITY ANALYSIS
# =============================================================================

SEEDS = [42, 123, 456]
MODEL_NAMES = ["MLP", "GCN", "SAGE", "SAGE_NoGraph"]

multi_seed_records = []

for seed in SEEDS:
    for model_name in MODEL_NAMES:

        saved = load_model_predictions(
            model_name=model_name,
            seed=seed,
        )

        y_true = np.asarray(
            saved["product_true"],
            dtype=float,
        )

        y_pred = np.asarray(
            saved["product_pred"],
            dtype=float,
        )

        multi_seed_records.append({
            "Seed": seed,
            "Model": (
                "GraphSAGE"
                if model_name == "SAGE"
                else model_name
            ),
            "RMSE": rmse(y_true, y_pred),
            "MAE": mae(y_true, y_pred),
            "R2": r2(y_true, y_pred),
        })


multi_seed_results = pd.DataFrame(
    multi_seed_records
)

display(multi_seed_results)

In [ ]:
# Calculate the Summary

multi_seed_summary = (
    multi_seed_results
    .groupby("Model")
    .agg(
        RMSE_Mean=("RMSE", "mean"),
        RMSE_SD=("RMSE", "std"),
        MAE_Mean=("MAE", "mean"),
        MAE_SD=("MAE", "std"),
        R2_Mean=("R2", "mean"),
        R2_SD=("R2", "std"),
    )
    .reset_index()
)

multi_seed_summary["RMSE Mean ± SD"] = (
    multi_seed_summary["RMSE_Mean"].map(
        lambda value: f"{value:.2f}"
    )
    + " ± "
    + multi_seed_summary["RMSE_SD"].map(
        lambda value: f"{value:.2f}"
    )
)

multi_seed_summary["MAE Mean ± SD"] = (
    multi_seed_summary["MAE_Mean"].map(
        lambda value: f"{value:.2f}"
    )
    + " ± "
    + multi_seed_summary["MAE_SD"].map(
        lambda value: f"{value:.2f}"
    )
)

multi_seed_summary["R² Mean ± SD"] = (
    multi_seed_summary["R2_Mean"].map(
        lambda value: f"{value:.4f}"
    )
    + " ± "
    + multi_seed_summary["R2_SD"].map(
        lambda value: f"{value:.4f}"
    )
)

publication_multi_seed_table = (
    multi_seed_summary[
        [
            "Model",
            "RMSE Mean ± SD",
            "MAE Mean ± SD",
            "R² Mean ± SD",
        ]
    ]
)

display(publication_multi_seed_table)

In [ ]:
#Save both the raw and summarised outputs:

raw_output_path = (
    ROBUSTNESS_DIR
    / "multi_seed_product_metrics_raw.csv"
)

summary_output_path = (
    ROBUSTNESS_DIR
    / "multi_seed_product_metrics_summary.csv"
)

multi_seed_results.to_csv(
    raw_output_path,
    index=False,
)

publication_multi_seed_table.to_csv(
    summary_output_path,
    index=False,
)

print("Raw results saved to:", raw_output_path)
print("Summary saved to:", summary_output_path)

In [ ]:

import sys
import importlib.metadata as metadata

print("Python:", sys.version)

packages = [
    "numpy",
    "pandas",
    "scipy",
    "scikit-learn",
    "statsmodels",
    "networkx",
    "matplotlib",
    "torch",
    "torch-geometric"
]

for package in packages:
    try:
        print(f"{package}=={metadata.version(package)}")
    except metadata.PackageNotFoundError:
        print(f"{package}: not installed")
